# Laboratorio 2: Data Augmentation
## ¿Cómo mejorar un modelo con datos artificiales?

### Objetivos
- Entender qué es **overfitting** y por qué ocurre
- Aprender a usar **Data Augmentation** con `ImageDataGenerator`
- Comparar el rendimiento de una CNN con y sin augmentation
- Experimentar interactivamente con diferentes transformaciones

### Concepto clave
Cuando tenemos **pocos datos**, el modelo tiende a **memorizar** las imágenes en vez de
aprender patrones generales. Esto se llama **overfitting**.

**Data Augmentation** genera variaciones artificiales (rotaciones, zoom, brillo, etc.)
para que el modelo vea más "diversidad" y aprenda mejor.


In [ ]:
# Instalar dependencias
!pip install -q tensorflow matplotlib seaborn ipywidgets scikit-learn

In [ ]:
import os
import shutil
import random
import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from sklearn.metrics import confusion_matrix, classification_report

import ipywidgets as widgets
from IPython.display import display, clear_output

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow: {tf.__version__}")

---
## 1. Preparación de Datos
Usamos la misma división del Lab 1.


In [ ]:
def split_dataset(dataset_dir, output_dir, split_ratio=0.8):
    """Divide el dataset en train y validation."""
    train_dir = os.path.join(output_dir, 'train')
    validation_dir = os.path.join(output_dir, 'validation')
    if os.path.exists(train_dir) and os.path.exists(validation_dir):
        print(f"La división ya existe en '{output_dir}'. Usando datos existentes.")
        return
    print(f"Creando división en '{output_dir}'...")
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(validation_dir, exist_ok=True)
    categories = ['glass', 'plastic']
    random.seed(42)
    for category in categories:
        category_dir = os.path.join(dataset_dir, category)
        images = os.listdir(category_dir)
        random.shuffle(images)
        train_size = int(len(images) * split_ratio)
        for subset, img_list in [('train', images[:train_size]),
                                  ('validation', images[train_size:])]:
            dest = os.path.join(output_dir, subset, category)
            os.makedirs(dest, exist_ok=True)
            for img in img_list:
                shutil.copy(os.path.join(category_dir, img), os.path.join(dest, img))
        print(f"  {category}: {train_size} train, {len(images) - train_size} validation")
    print("División completada.")

dataset_dir = '../data/botellas'
output_dir = '../data/botellas_split'
split_dataset(dataset_dir, output_dir, split_ratio=0.8)

---
## 2. ¿Qué es el Overfitting?

Cuando un modelo tiene **alta precisión en entrenamiento** pero **baja en validación**,
está **memorizando** los datos en vez de aprender patrones generales.

| | Train Accuracy | Val Accuracy | Estado |
|---|---|---|---|
| Buen modelo | 95% | 92% | Generaliza bien |
| Overfitting | 99% | 70% | Memoriza, no aprende |
| Underfitting | 60% | 58% | No aprendió nada |

**Solución principal: Data Augmentation** → generar variaciones para que el modelo
vea más diversidad de imágenes.


---
## 3. Experimentar con Data Augmentation (Interactivo)

El `ImageDataGenerator` de Keras genera variaciones **en tiempo real** durante el entrenamiento.
No modifica las imágenes originales, solo las transforma al vuelo.

### Mueva los sliders para ver el efecto de cada transformación:


In [ ]:
def preview_augmentation():
    """Widget interactivo para experimentar con Data Augmentation."""
    ejemplo_dir = os.path.join(output_dir, 'train', 'glass')
    imgs = [f for f in os.listdir(ejemplo_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if not imgs:
        print("No hay imágenes disponibles")
        return
    ejemplo_path = os.path.join(ejemplo_dir, imgs[0])
    ejemplo_array = img_to_array(load_img(ejemplo_path, target_size=(64, 64))) / 255.0

    sw = {'description_width': '120px'}
    rotacion  = widgets.IntSlider(min=0, max=90, step=5, value=0,
                                  description='Rotación (°):', style=sw)
    shift_h   = widgets.FloatSlider(min=0, max=0.3, step=0.05, value=0,
                                    description='Desplaz. Horiz:', style=sw)
    shift_v   = widgets.FloatSlider(min=0, max=0.3, step=0.05, value=0,
                                    description='Desplaz. Vert:', style=sw)
    shear     = widgets.FloatSlider(min=0, max=0.3, step=0.05, value=0,
                                    description='Inclinación:', style=sw)
    zoom      = widgets.FloatSlider(min=0, max=0.5, step=0.05, value=0,
                                    description='Zoom:', style=sw)
    flip_h    = widgets.Checkbox(value=False, description='Flip horizontal')

    output = widgets.Output()

    def actualizar(_=None):
        with output:
            clear_output(wait=True)
            datagen = ImageDataGenerator(
                rotation_range=rotacion.value,
                width_shift_range=shift_h.value,
                height_shift_range=shift_v.value,
                shear_range=shear.value,
                zoom_range=zoom.value,
                horizontal_flip=flip_h.value,
                fill_mode='nearest'
            )
            fig, axes = plt.subplots(1, 6, figsize=(20, 3.5))
            axes[0].imshow(ejemplo_array)
            axes[0].set_title("ORIGINAL", fontweight='bold', color='blue')
            axes[0].axis('off')

            batch = np.expand_dims(ejemplo_array, 0)
            gen = datagen.flow(batch, batch_size=1)
            for i in range(1, 6):
                aug = next(gen)[0]
                axes[i].imshow(np.clip(aug, 0, 1))
                axes[i].set_title(f"Variación {i}")
                axes[i].axis('off')
            plt.tight_layout()
            plt.show()

    for w in [rotacion, shift_h, shift_v, shear, zoom, flip_h]:
        w.observe(actualizar, names='value')

    sliders = widgets.VBox([rotacion, shift_h, shift_v, shear, zoom, flip_h])
    display(widgets.HBox([sliders, output]))
    actualizar()

preview_augmentation()

**Observe:**
- `rotation_range`: rota la imagen aleatoriamente
- `width/height_shift_range`: desplaza la imagen
- `shear_range`: inclina la imagen
- `zoom_range`: hace zoom aleatorio
- `horizontal_flip`: voltea horizontalmente

> Cada vez que el modelo entrena, ve una variación **diferente** de cada imagen.
> Así, con 100 imágenes originales, el modelo puede ver miles de variaciones.


---
## 4. Entrenamiento con Data Augmentation

Usamos la **misma arquitectura** del Lab 1, pero ahora con Data Augmentation
en el generador de entrenamiento.


In [ ]:
# Misma arquitectura que Lab 1
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(64, 64, 3)),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(2, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

print("Modelo creado (misma arquitectura que Lab 1)")
model.summary()

In [ ]:
# Data Augmentation en el generador de entrenamiento
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,         # Rota hasta 20°
    width_shift_range=0.1,     # Desplaza horizontalmente 10%
    height_shift_range=0.1,    # Desplaza verticalmente 10%
    shear_range=0.1,           # Inclina la imagen
    zoom_range=0.2,            # Zoom aleatorio hasta 20%
    horizontal_flip=True,      # Voltea horizontalmente
    fill_mode='nearest'        # Rellena píxeles nuevos
)

# Validación: SOLO normalizar (nunca augmentation)
validation_datagen = ImageDataGenerator(rescale=1./255)

train_set = train_datagen.flow_from_directory(
    os.path.join(output_dir, 'train'),
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical'
)

validation_set = validation_datagen.flow_from_directory(
    os.path.join(output_dir, 'validation'),
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical'
)

### Comparación: imágenes originales vs augmentadas


In [ ]:
# Comparar batch original vs augmentado
original_gen = ImageDataGenerator(rescale=1./255)
augmented_gen = train_datagen

orig_set = original_gen.flow_from_directory(
    os.path.join(output_dir, 'train'),
    target_size=(64, 64), batch_size=4, class_mode='categorical', shuffle=False)
aug_set = augmented_gen.flow_from_directory(
    os.path.join(output_dir, 'train'),
    target_size=(64, 64), batch_size=4, class_mode='categorical', shuffle=False)

orig_batch, _ = next(orig_set)
aug_batch, _ = next(aug_set)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    axes[0][i].imshow(orig_batch[i])
    axes[0][i].set_title("Original" if i == 0 else "")
    axes[0][i].axis('off')
    axes[1][i].imshow(np.clip(aug_batch[i], 0, 1))
    axes[1][i].set_title("Con augmentation" if i == 0 else "")
    axes[1][i].axis('off')

plt.suptitle('Fila superior: originales | Fila inferior: con data augmentation', fontsize=14)
plt.tight_layout()
plt.show()

### Entrenamiento


In [ ]:
print("Entrenando con Data Augmentation...")
history = model.fit(
    train_set,
    steps_per_epoch=train_set.samples // train_set.batch_size,
    epochs=100,
    validation_data=validation_set,
    validation_steps=validation_set.samples // validation_set.batch_size
)

loss, accuracy = model.evaluate(validation_set)
print(f'\nResultado final → Loss: {loss:.4f}, Accuracy: {accuracy:.4f}')

In [ ]:
# Curvas de entrenamiento
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history.history['accuracy']) + 1)

ax1.plot(epochs_range, history.history['accuracy'], 'b-', label='Train', linewidth=2)
ax1.plot(epochs_range, history.history['val_accuracy'], 'r-', label='Validation', linewidth=2)
ax1.set_title('Precisión')
ax1.set_xlabel('Época')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history.history['loss'], 'b-', label='Train', linewidth=2)
ax2.plot(epochs_range, history.history['val_loss'], 'r-', label='Validation', linewidth=2)
ax2.set_title('Pérdida')
ax2.set_xlabel('Época')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Detectar overfitting
gap = history.history['accuracy'][-1] - history.history['val_accuracy'][-1]
if gap > 0.15:
    fig.suptitle(f'⚠ Overfitting detectado (gap: {gap:.1%})', color='red', fontsize=14)
else:
    fig.suptitle(f'Entrenamiento con Data Augmentation (gap: {gap:.1%})', fontsize=14)

plt.tight_layout()
plt.show()

**Compare con los resultados del Lab 1:**
- ¿Las curvas de train y validation están más juntas que en el Lab 1?
- ¿El overfitting se redujo?
- ¿La accuracy de validación mejoró?

**Respuesta:** *(Escriba aquí)*


---
## 5. Evaluación


In [ ]:
validation_set.reset()
predictions = model.predict(validation_set, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = validation_set.classes
class_names = list(validation_set.class_indices.keys())

print("Reporte de Clasificación:")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title("Matriz de Confusión (con Data Augmentation)")
plt.tight_layout()
plt.show()

---
## 6. Predicciones Interactivas


In [ ]:
class_indices = train_set.class_indices
class_labels = {v: k for k, v in class_indices.items()}

def predictor_interactivo():
    btn = widgets.Button(description='Predecir imagen aleatoria',
                         icon='search', button_style='success')
    output = widgets.Output()

    val_imgs = []
    for ext in ('*.jpg', '*.jpeg', '*.png'):
        val_imgs += glob.glob(os.path.join(output_dir, 'validation', '*', ext))

    def predecir(_=None):
        with output:
            clear_output(wait=True)
            if not val_imgs:
                print("No hay imágenes")
                return
            img_path = random.choice(val_imgs)
            real = os.path.basename(os.path.dirname(img_path))

            img = load_img(img_path, target_size=(64, 64))
            arr = img_to_array(img) / 255.0
            pred = model.predict(np.expand_dims(arr, 0), verbose=0)[0]
            idx = np.argmax(pred)
            label = class_labels[idx]

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4),
                                            gridspec_kw={'width_ratios': [1, 1.2]})
            ax1.imshow(img)
            ok = label == real
            ax1.set_title(f"{'✓' if ok else '✗'} Real: {real} | Pred: {label}",
                          color='green' if ok else 'red', fontweight='bold')
            ax1.axis('off')

            names = list(class_labels.values())
            colors = ['#2ecc71' if n == label else '#bdc3c7' for n in names]
            ax2.barh(names, pred, color=colors)
            ax2.set_xlim(0, 1)
            ax2.set_xlabel('Confianza')
            plt.tight_layout()
            plt.show()

    btn.on_click(predecir)
    display(btn)
    display(output)
    predecir()

predictor_interactivo()

---
## 7. Exportar Modelo


In [ ]:
model.save("./modelo_lab2_aug.h5")
print("Modelo exportado como: modelo_lab2_aug.h5")

---
## Resumen

| | Lab 1 (sin augmentation) | Lab 2 (con augmentation) |
|---|---|---|
| Datos de entrenamiento | Imágenes originales | Variaciones aleatorias |
| Riesgo de overfitting | Alto | Reducido |
| Generalización | Baja | Mejor |

**Conclusión:** Data Augmentation es una técnica simple pero poderosa que mejora la
capacidad del modelo para generalizar a imágenes nuevas.

En el **Lab 3**, aplicaremos estos conceptos a su proyecto de tapitas.
